In [2]:
import pandas as pd
import db_dtypes
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_validate, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support, brier_score_loss
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier

In [8]:
def build_preprocessor(df_raw):

    analysis_cols = [
    'patient_age', 'gender', 'length_of_stay', 'main_code', 'num_diagnoses',
    'num_chronic_conditions', 'num_procedures', 'has_diabetes', 'has_cancer',
    'has_hiv', 'has_hf', 'has_alz', 'has_ckd', 'had_surgery', 'admission_cost',
    'total_procedure_costs', 'total_medication_costs', 'total_stay_cost', 
    'admissions_365d', 'tot_length_of_stay_365d', 'avg_cost_of_prev_stays',
    'is_planned', 'following_unplanned_admission_flag', 'readmit_30d', 'readmit_90d'
    ]

    df_numeric = df_raw[analysis_cols]

    df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
    df_numeric['following_unplanned_admission_flag'] = df_numeric['following_unplanned_admission_flag'].fillna(0)

    df_numeric = pd.get_dummies(df_numeric)
    df_numeric = df_numeric.drop(columns = 'gender_F')

    mask = df_numeric['following_unplanned_admission_flag'] == 0
    df_numeric.loc[mask, ['readmit_30d', 'readmit_90d']] = 0
    mask = df_numeric['readmit_90d'] == 0
    df_numeric.loc[mask, 'following_unplanned_admission_flag'] = 0

    df_results = df_numeric[['readmit_30d', 'readmit_90d']]
    df_numeric['log_stay_cost'] = np.log(df_numeric['total_stay_cost'])
    df_numeric['log_prev_avg_stay_cost'] = np.log1p(df_numeric['avg_cost_of_prev_stays'])
    df_numeric['log_total_procedure_costs'] = np.log1p(df_numeric['total_procedure_costs'])
    df_numeric['log_total_medication_costs'] = np.log1p(df_numeric['total_medication_costs'])

    df_numeric = df_numeric.drop(columns = ['total_stay_cost', 'avg_cost_of_prev_stays', 'total_procedure_costs', 'total_medication_costs'
,'readmit_30d', 'readmit_90d', 'following_unplanned_admission_flag', 'main_code'])

    return df_numeric, df_results

In [9]:
def make_train_test_split(X, y):

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 0)

    return X_train, X_test, y_train, y_test

In [10]:
df_raw = pd.read_csv("D:\Python Projects\Hospital readmission risk\index_stay.csv", sep = ',')
df_numeric, df_results = build_preprocessor(df_raw)

C:\Users\4infi\AppData\Local\Temp\ipykernel_30068\3613884750.py:1: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv("D:\Python Projects\Hospital readmission risk\index_stay.csv", sep = ',')
C:\Users\4infi\AppData\Local\Temp\ipykernel_30068\2235083803.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
C:\Users\4infi\AppData\Local\Temp\ipykernel_30068\2235083803.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-doc

In [11]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])

In [12]:
scaler = StandardScaler()

data = scaler.fit_transform(X_train)

In [16]:
rf = RandomForestClassifier()

rf_param_dist = {
    "n_estimators": [100, 200, 300, 500, 800, 1000],
    "max_depth": [None, 4, 6, 8, 10],
    "min_samples_split": [2, 10, 20, 50, 100],
    "min_samples_leaf": [1, 5, 10, 20, 50],
    "max_features": ["sqrt", 0.3, 0.5],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

best_rf_params_90 = rf_search.best_params_
best_rf_score_90 = rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [17]:
best_rf_params_90

{'n_estimators': 300,
 'min_samples_split': 2,
 'min_samples_leaf': 5,
 'max_features': 0.5,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

In [18]:
best_rf_score_90

np.float64(0.8257399183499462)

In [27]:
rf = RandomForestClassifier(random_state = 42, **best_rf_params_90)

rf.fit(data, y_train)

test_data = scaler.fit_transform(X_test)

y_pred = rf.predict_proba(test_data)[:,1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

0.8480981381370178 0.9544017602477638


In [19]:
rf = RandomForestClassifier()

In [20]:
rf_param_dist = {
    "n_estimators": [250, 300, 350],
    "max_depth": [None, 1],
    "min_samples_split": [6, 10, 15],
    "min_samples_leaf": [3, 5, 7],
    "max_features": [0.4, 0.5, 0.6],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


np.float64(0.8267943538449747)

In [ ]:
rf_search.best_params_

{'n_estimators': 250,
 'min_samples_split': 10,
 'min_samples_leaf': 5,
 'max_features': 0.6,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

In [28]:
rf = RandomForestClassifier(random_state = 42, **rf_search.best_params_)

rf.fit(data, y_train)

y_pred = rf.predict_proba(test_data)[:,1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

0.8486900766231116 0.9541183911437792


---

In [16]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
data = scaler.fit_transform(X_train)

reunite above and below into one cage

In [ ]:

rf = RandomForestClassifier()

rf_param_dist = {
    "n_estimators": [100, 200, 300, 400, 500],
    "max_depth": [None, 4, 10],
    "min_samples_split": [5, 10, 20],
    "min_samples_leaf": [2, 5, 10],
    "max_features": ["sqrt", 0.3, 0.5, 0.6, 0.7],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

best_rf_params_30 = rf_search.best_params_
best_rf_score_30 = rf_search.best_score_

Fitting 3 folds for each of 30 candidates, totalling 90 fits


In [30]:
best_rf_score_30

np.float64(0.6260811606513488)

In [ ]:
best_rf_params_30

{'n_estimators': 500,
 'min_samples_split': 20,
 'min_samples_leaf': 5,
 'max_features': 0.6,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

: 

In [17]:
rf = RandomForestClassifier(random_state = 42, **{'n_estimators': 500,
 'min_samples_split': 20,
 'min_samples_leaf': 5,
 'max_features': 0.6,
 'max_depth': None,
 'class_weight': 'balanced_subsample'})

test_data = scaler.fit_transform(X_test)

rf.fit(data, y_train)

y_pred = rf.predict_proba(test_data)[:,1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

0.6392579427898621 0.9475224785411042


In [18]:
rf = RandomForestClassifier()

rf_param_dist = {
    "n_estimators": [400, 500, 600],
    "max_depth": [None, 4],
    "min_samples_split": [10, 20, 40],
    "min_samples_leaf": [5, 10, 20],
    "max_features": [0.5, 0.6, 0.7],
    "class_weight": ["balanced_subsample"],
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring="average_precision",   # or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(data, y_train)

Fitting 3 folds for each of 30 candidates, totalling 90 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'class_weight': ['balanced_subsample'], 'max_depth': [None, 4], 'max_features': [0.5, 0.6, ...], 'min_samples_leaf': [5, 10, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be u

In [19]:
rf_search.best_params_

{'n_estimators': 600,
 'min_samples_split': 10,
 'min_samples_leaf': 5,
 'max_features': 0.7,
 'max_depth': None,
 'class_weight': 'balanced_subsample'}

In [20]:
rf_search.best_score_

np.float64(0.6270597004387395)

In [21]:
rf = RandomForestClassifier(random_state = 42, **rf_search.best_params_)

test_data = scaler.fit_transform(X_test)

rf.fit(data, y_train)

y_pred = rf.predict_proba(test_data)[:,1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

0.6382197124661948 0.9455539557082621


---

In [22]:
rf = RandomForestClassifier(random_state = 42, **{'n_estimators': 250,
 'min_samples_split': 10,
 'min_samples_leaf': 5,
 'max_features': 0.6,
 'max_depth': None,
 'class_weight': 'balanced_subsample'})

test_data = scaler.fit_transform(X_test)

rf.fit(data, y_train)

y_pred = rf.predict_proba(test_data)[:,1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

0.6382304709402047 0.9461745488327638


---

In [23]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])
data = scaler.fit_transform(X_train)

In [24]:
lgbm = LGBMClassifier(objective="binary", force_col_wise = True, random_state=42)

lgbm_param_dist = {
    "n_estimators": [100, 200, 400, 600, 1000],
    "learning_rate": [0.03, 0.05, 0.1],
    "num_leaves": [31, 63, 127],
    "max_depth": [-1, 6, 8, 10],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9],
    "min_child_samples": [10, 20, 40],
    "reg_lambda": [0.0, 1.0, 5.0],
    "reg_alpha": [0.0, 0.5, 1.0],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

lgbm_search.fit(data, y_train)  # same 30d training set or a stratified subset

best_lgbm_params_90 = lgbm_search.best_params_
best_lgbm_score_90 = lgbm_search.best_score_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 7025, number of negative: 69627
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.091648 -> initscore=-2.293677
[LightGBM] [Info] Start training from score -2.293677
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [25]:
best_lgbm_params_90

{'subsample': 0.8,
 'reg_lambda': 0.0,
 'reg_alpha': 1.0,
 'num_leaves': 127,
 'n_estimators': 200,
 'min_child_samples': 10,
 'max_depth': 10,
 'learning_rate': 0.05,
 'colsample_bytree': 0.9}

In [26]:
best_lgbm_score_90

np.float64(0.83468382747839)

In [27]:
lgb_general = LGBMClassifier(objective="binary", force_col_wise = True, **best_lgbm_params_90)

test_data = scaler.fit_transform(X_test)

lgb_general.fit(test_data, y_test)

y_pred = lgb_general.predict_proba(test_data)[:, 1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 3024, number of negative: 29827
[LightGBM] [Info] Total Bins 1396
[LightGBM] [Info] Number of data points in the train set: 32851, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.092052 -> initscore=-2.288834
[LightGBM] [Info] Start training from score -2.288834
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [28]:
lgbm_param_dist = {
    "n_estimators": [150, 200, 250],
    "learning_rate": [0.03, 0.05, 0.07],
    "num_leaves": [90, 127, 160],
    "max_depth": [-1, 8, 10, 12],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.85, 0.9, 0.95],
    "min_child_samples": [5, 10, 15],
    "reg_lambda": [0.0, 0.25, 0.5],
    "reg_alpha": [0.5, 1, 2, 5]
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

lgbm_search.fit(data, y_train)

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 7025, number of negative: 69627
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.091648 -> initscore=-2.293677
[LightGBM] [Info] Start training from score -2.293677


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.85, 0.9, ...], 'learning_rate': [0.03, 0.05, ...], 'max_depth': [-1, 8, ...], 'min_child_samples': [5, 10, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies th

In [29]:
lgbm_search.best_score_

np.float64(0.8351891593490843)

In [30]:
lgbm_search.best_params_

{'subsample': 0.7,
 'reg_lambda': 0.25,
 'reg_alpha': 0.5,
 'num_leaves': 127,
 'n_estimators': 200,
 'min_child_samples': 10,
 'max_depth': -1,
 'learning_rate': 0.05,
 'colsample_bytree': 0.9}

In [31]:
lgb_general = LGBMClassifier(objective="binary", force_col_wise = True, **lgbm_search.best_params_)

test_data = scaler.fit_transform(X_test)

lgb_general.fit(test_data, y_test)

y_pred = lgb_general.predict_proba(test_data)[:, 1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 3024, number of negative: 29827
[LightGBM] [Info] Total Bins 1396
[LightGBM] [Info] Number of data points in the train set: 32851, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.092052 -> initscore=-2.288834
[LightGBM] [Info] Start training from score -2.288834
0.9863985592211929 0.9980972727561389


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


---

In [32]:
X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
data = scaler.fit_transform(X_train)

lgbm_param_dist = {
    "n_estimators": [50, 100, 200, 300, 400],
    "learning_rate": [0.01, 0.03, 0.05, 0.07],
    "num_leaves": [63, 100, 128, 180, 256],
    "max_depth": [-1, 6, 8, 10],
    "subsample": [0.5, 0.6, 0.7, 0.8],
    "colsample_bytree": [0.5, 0.6, 0.7, 0.8],
    "min_child_samples": [2, 5, 10, 20],
    "reg_lambda": [0.0, 0.5, 1],
    "reg_alpha": [0.0, 1, 5, 10],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lgbm_search.fit(data, y_train)  # same 30d training set or a stratified subset

best_lgbm_params_30 = lgbm_search.best_params_
best_lgbm_score_30 = lgbm_search.best_score_

Fitting 3 folds for each of 40 candidates, totalling 120 fits
[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [33]:
best_lgbm_params_30


{'subsample': 0.8,
 'reg_lambda': 0.0,
 'reg_alpha': 5,
 'num_leaves': 256,
 'n_estimators': 400,
 'min_child_samples': 5,
 'max_depth': 8,
 'learning_rate': 0.03,
 'colsample_bytree': 0.7}

In [34]:
best_lgbm_score_30

np.float64(0.6418547788691705)

In [35]:
lgb_general = LGBMClassifier(objective="binary", force_col_wise = True, **best_lgbm_params_30)

test_data = scaler.fit_transform(X_test)

lgb_general.fit(test_data, y_test)

y_pred = lgb_general.predict_proba(test_data)[:, 1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 1551, number of negative: 31300
[LightGBM] [Info] Total Bins 1396
[LightGBM] [Info] Number of data points in the train set: 32851, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047213 -> initscore=-3.004718
[LightGBM] [Info] Start training from score -3.004718
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.8124748257996871 0.9760255570455421


In [36]:
lgbm_param_dist = {
    "n_estimators": [300, 400, 500],
    "learning_rate": [0.01, 0.03, 0.05],
    "num_leaves": [200, 256, 312, 380],
    "max_depth": [-1, 6, 8, 10],
    "subsample": [0,7, 0.8, 0.9],
    "colsample_bytree": [0.6, 0.7, 0.8],
    "min_child_samples": [3, 5, 7, 10],
    "reg_lambda": [0.0, 0.25, 0.5],
    "reg_alpha": [3, 5, 7, 10],
}

lgbm_search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_dist,
    n_iter=40,
    scoring="average_precision",   # again: or "roc_auc"
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

lgbm_search.fit(data, y_train)  # same 30d training set or a stratified subset

Fitting 3 folds for each of 40 candidates, totalling 120 fits


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
45 fits failed out of a total of 120.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
18 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1560, in fit
    super().fit(
  File "d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\lightgbm\sklearn.py", line 1049, in fit
    se

[LightGBM] [Info] Number of positive: 3610, number of negative: 73042
[LightGBM] [Info] Total Bins 1428
[LightGBM] [Info] Number of data points in the train set: 76652, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047096 -> initscore=-3.007327
[LightGBM] [Info] Start training from score -3.007327
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LGBMClassifie...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.7, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [-1, 6, ...], 'min_child_samples': [3, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",40
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that

In [37]:
lgbm_search.best_params_

{'subsample': 0.8,
 'reg_lambda': 0.25,
 'reg_alpha': 3,
 'num_leaves': 256,
 'n_estimators': 500,
 'min_child_samples': 7,
 'max_depth': -1,
 'learning_rate': 0.01,
 'colsample_bytree': 0.8}

In [38]:
lgbm_search.best_score_

np.float64(0.6422212562979771)

In [39]:
lgb_general = LGBMClassifier(objective="binary", force_col_wise = True, **lgbm_search.best_params_)

test_data = scaler.fit_transform(X_test)

lgb_general.fit(test_data, y_test)

y_pred = lgb_general.predict_proba(test_data)[:, 1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 1551, number of negative: 31300
[LightGBM] [Info] Total Bins 1396
[LightGBM] [Info] Number of data points in the train set: 32851, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047213 -> initscore=-3.004718
[LightGBM] [Info] Start training from score -3.004718
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


0.8439811650751595 0.9820440177727242


In [40]:
lgbm_params = {
    'subsample': 0.7,
 'reg_lambda': 0.25,
 'reg_alpha': 0.5,
 'num_leaves': 127,
 'n_estimators': 200,
 'min_child_samples': 10,
 'max_depth': -1,
 'learning_rate': 0.05,
 'colsample_bytree': 0.9
}

lgb_general_new = LGBMClassifier(objective="binary", force_col_wise = True, **lgbm_params)

test_data = scaler.fit_transform(X_test)

lgb_general_new.fit(test_data, y_test)

y_pred = lgb_general_new.predict_proba(test_data)[:, 1]

print(average_precision_score(y_test, y_pred), roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 1551, number of negative: 31300
[LightGBM] [Info] Total Bins 1396
[LightGBM] [Info] Number of data points in the train set: 32851, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047213 -> initscore=-3.004718
[LightGBM] [Info] Start training from score -3.004718
0.9829634660667963 0.9987242591093451


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from hospital_readmission_risk.data import load_data
import pandas as pd
from hospital_readmission_risk.preprocessing import preprocess_data
from hospital_readmission_risk.config import train_data_path, train_sql, models
from hospital_readmission_risk.models import build_pipeline, model_config_builder
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from hospital_readmission_risk.hyperparameter_optimizer import randomize_search

ModuleNotFoundError: No module named 'data'

In [ ]:
df_train = load_data(train_data_path, train_sql, query = False)
X_train, y_train = preprocess_data(df_train)
start_distribution = {
    "logreg" : {
            "class_weight": ["balanced"],
            "solver": ["saga"],
            "max_iter": [100, 500, 1000, 2000]
    },
    "rf" : {
            "n_estimators": [100, 200, 300, 500, 800, 1000],
            "max_depth": [None, 4, 6, 8, 10],
            "min_samples_split": [2, 10, 20, 50, 100],
            "min_samples_leaf": [1, 5, 10, 20, 50],
            "max_features": ["sqrt", 0.3, 0.5],
            "class_weight": ["balanced_subsample"]
    },
    "lightgbm" : {
            "n_estimators": [50, 100, 200, 300, 400],
            "learning_rate": [0.01, 0.03, 0.05, 0.07],
            "num_leaves": [63, 100, 128, 180, 256],
            "max_depth": [-1, 6, 8, 10],
            "subsample": [0.5, 0.6, 0.7, 0.8],
            "colsample_bytree": [0.5, 0.6, 0.7, 0.8],
            "min_child_samples": [2, 5, 10, 20],
            "reg_lambda": [0.0, 0.5, 1],
            "reg_alpha": [0.0, 1, 5, 10],
    }
}
models_built = model_config_builder(models, set_params = False)
optimal_params : dict[str, dict[str, float]] = {}
for model in models_built:
    distribution = start_distribution[model]
    pipe = build_pipeline(model, models_built[model])
    best_params, best_score = randomize_search(pipe, distribution, X_train, y_train, random_state = 42)
    optimal_params[model] = best_params
    print(f"{model}: best score is {best_score}")
